In [1]:
import duckdb

db = duckdb.connect()
db.execute("CREATE TABLE Client (id_client INT PRIMARY KEY, nb_reserved_seats INT NOT NULL, balance INT NOT NULL)")
db.execute("INSERT INTO Client VALUES (1, 0, 2000), (2, 0, 350)")
booking1, booking2 = db.cursor(), db.cursor()

In [2]:
def attempt(con, sql):
    try:
        con.execute(sql)
        print(f"{sql} -> OK")
    except duckdb.TransactionException as exc:
        print(f"{sql} -> {type(exc).__name__}: {exc}")

attempt(booking1, "BEGIN")
attempt(booking1, "UPDATE Client SET nb_reserved_seats = nb_reserved_seats + 5 WHERE id_client=1")
print("booking1 sees:", booking1.execute("SELECT * FROM Client WHERE id_client=1").fetchall())
attempt(booking2, "BEGIN")
attempt(booking2, "UPDATE Client SET nb_reserved_seats = nb_reserved_seats + 5 WHERE id_client=1")

BEGIN -> OK
UPDATE Client SET nb_reserved_seats = nb_reserved_seats + 5 WHERE id_client=1 -> OK
booking1 sees: [(1, 5, 2000)]
BEGIN -> OK
UPDATE Client SET nb_reserved_seats = nb_reserved_seats + 5 WHERE id_client=1 -> TransactionException: TransactionContext Error: Conflict on update!


In [3]:
def book(con, client, seats):
    for attempt in range(4):
        try:
            con.execute("BEGIN")
            con.execute(
                "UPDATE Client SET nb_reserved_seats = nb_reserved_seats + ? WHERE id_client = ?",
                [seats, client],
            )
            con.execute("COMMIT")
            return
        except duckdb.TransactionException:
            con.execute("ROLLBACK")
            if attempt == 3:
                raise

for con in (booking1, booking2):
    try:
        con.execute("ROLLBACK")
    except duckdb.TransactionException:
        pass

In [4]:
db.execute("DROP TABLE Client")
db.execute("CREATE TABLE Client (id_client INT PRIMARY KEY, nb_reserved_seats INT NOT NULL, balance INT NOT NULL)")
db.execute("INSERT INTO Client VALUES (1, 0, 2000), (2, 0, 350)")

attempt(booking1, "BEGIN")
attempt(booking2, "BEGIN")
print("booking1 reads client 2:", booking1.execute("SELECT nb_reserved_seats FROM Client WHERE id_client=2").fetchone())
print("booking2 reads client 1:", booking2.execute("SELECT nb_reserved_seats FROM Client WHERE id_client=1").fetchone())
attempt(booking1, "UPDATE Client SET nb_reserved_seats = 2 WHERE id_client=1")
attempt(booking2, "UPDATE Client SET nb_reserved_seats = 2 WHERE id_client=2")
attempt(booking1, "COMMIT")
attempt(booking2, "COMMIT")
for con in (booking1, booking2):
    try:
        con.execute("ROLLBACK")
    except duckdb.TransactionException:
        pass
print("Final Client rows:", db.execute("SELECT * FROM Client ORDER BY id_client").fetchall())

BEGIN -> OK
BEGIN -> OK
booking1 reads client 2: (0,)
booking2 reads client 1: (0,)
UPDATE Client SET nb_reserved_seats = 2 WHERE id_client=1 -> OK
UPDATE Client SET nb_reserved_seats = 2 WHERE id_client=2 -> OK
COMMIT -> OK
COMMIT -> OK
Final Client rows: [(1, 2, 2000), (2, 2, 350)]


In [5]:
db.execute("DROP TABLE Client")
db.execute("CREATE TABLE Client (id_client INT PRIMARY KEY, nb_reserved_seats INT NOT NULL, balance INT NOT NULL)")
db.execute("INSERT INTO Client VALUES (1, 0, 2000), (2, 0, 350)")
book(booking1, 1, 5)
print("After book(booking1, 1, 5):", db.execute("SELECT * FROM Client WHERE id_client=1").fetchone())

After book(booking1, 1, 5): (1, 5, 2000)
